# Activity Recognition System: the complete code
**Project 24 · IBM Skill Lab · Review 2.** Every code file in the project, in the order the data flows, with who wrote it. The code is shown exactly as it is in the repository.

In Colab, run the first cell: it clones the repository, so every file also appears in the **Files** panel on the left (`activity-recognition/`), where you can open it. The notebooks `01`–`10` hold the experiments, and `review_1`–`review_5` run each part.

| Part | Files (owner) |
|---|---|
| Settings | `config/config.yaml` (everyone) |
| Data and preprocessing | `src/data_loader.py` (Nihal), `src/preprocess.py` (Nihal) |
| Validation | `src/validation.py` (Shravya) |
| Models, training, evaluation, phone export | `src/models.py` (Yash), `src/train.py` (Yash), `src/evaluate.py` (Yash), `src/export.py` (Yash) |
| Prediction and live phone | `src/inference.py` (Yashaswini), `src/live.py` (Yashaswini) |
| Camera pipeline | `src/video_pose.py` (Yyug), `notebooks/09_stairs_direction.py` (Yyug), `notebooks/10_body_points.py` (Yyug) |
| Web app (Streamlit) | `app/streamlit_app.py` (Yashaswini), `app/overview.py` (Yashaswini), `app/home.py` (Yashaswini), `app/pages/1_Live.py` (Yashaswini), `app/pages/2_Video.py` (Yashaswini), `app/components/ui.py` (Yashaswini) |
| Tests | `tests/test_validation.py` (Shravya), `tests/test_load_own.py` (Nihal), `tests/test_smooth.py` (Yashaswini), `tests/test_live.py` (Yashaswini), `tests/test_video_pose.py` (Yyug), `tests/test_ui_helpers.py` (Yashaswini) |

In [1]:
import os, sys, pathlib, subprocess
if "google.colab" in sys.modules:
    if not os.path.exists("/content/activity-recognition"):
        subprocess.run(["git", "clone", "-q", "https://github.com/rajyyug1132/activity-recognition", "/content/activity-recognition"], check=True)
    os.chdir("/content/activity-recognition")
elif pathlib.Path.cwd().name == "notebooks":
    os.chdir("..")
from IPython.display import Code, Markdown, display

def show_file(path):
    text = open(path, encoding="utf-8").read()
    display(Markdown(f"`{path}` · {text.count(chr(10)) + 1} lines"))
    display(Code(text, language="yaml" if path.endswith(".yaml") else "python"))
print("files found:", sum(1 for p in pathlib.Path(".").rglob("*.py") if ".venv" not in str(p)))

files found: 27


## Settings

### `config/config.yaml` · everyone

In [2]:
show_file("config/config.yaml")

`config/config.yaml` · 22 lines

sampling_rate_hz: 50
window_samples: 128      # 2.56 s at 50 Hz
overlap: 0.5
noise_cutoff_hz: 20
gravity_cutoff_hz: 0.3
activities: [WALKING, WALKING_UPSTAIRS, WALKING_DOWNSTAIRS, SITTING, STANDING, LAYING]
split:
  uci: official          # 21 train subjects, 9 test subjects
  uci_val: [3, 22, 25]   # held out from UCI train by person (seed 42)
  own_train: [p1, p2, p3]
  own_test: [p4, p5]
seed: 42
norm_stats: config/norm_stats.npz   # train-only per-channel mean/std
smooth_window: 5       # majority-vote smoothing window (odd, in windows)
video:                   # camera pipeline (src/video_pose.py)
  fps: 25
  window_frames: 50      # 2 s at 25 fps
  overlap: 0.5
  max_gap: 5             # NaN gaps up to this many frames are interpolated
  pose_model: models/pose_landmarker_full.task   # MediaPipe .task file (gitignored)
  classes: [walking, standing, stairs]   # camera model: no sitting/lying clips, up/down stairs merged

## Data and preprocessing

### `src/data_loader.py` · Nihal

In [3]:
show_file("src/data_loader.py")

`src/data_loader.py` · 73 lines

"""Owner: Nihal. Load the UCI HAR dataset in one call."""
import numpy as np

SIGNALS = ["body_acc_x", "body_acc_y", "body_acc_z",
           "body_gyro_x", "body_gyro_y", "body_gyro_z",
           "total_acc_x", "total_acc_y", "total_acc_z"]


def _raw(root, split):
    files = [f"{root}/{split}/Inertial Signals/{s}_{split}.txt" for s in SIGNALS]
    return np.stack([np.loadtxt(f) for f in files], axis=-1)   # [N, 128, 9]


def load_uci(root) -> dict:
    """Returns X_raw [N,128,9], X_feat [N,561], y [N] (0-5), subject [N] for train and test.

    Shape: {"train": {"X_raw", "X_feat", "y", "subject"}, "test": {...}}.
    `root` is the unzipped "UCI HAR Dataset" folder.
    """
    out = {}
    for split in ("train", "test"):
        out[split] = {
            "X_raw": _raw(root, split),
            "X_feat": np.loadtxt(f"{root}/{split}/X_{split}.txt"),
            "y": np.loadtxt(f"{root}/{split}/y_{split}.txt", dtype=int) - 1,
            "subject": np.loadtxt(f"{root}/{split}/subject_{split}.txt", dtype=int),
        }
    return out


def split_val(train, val_subjects):
    """Split the train dict by person (never by row). Returns (train, val)."""
    m = np.isin(train["subject"], val_subjects)
    return ({k: v[~m] for k, v in train.items()},
            {k: v[m] for k, v in train.items()})


OWN_ACTS = ["walking", "walking_upstairs", "walking_downstairs", "sitting", "standing", "lying"]  # label 0-5


def load_own(root, cfg) -> dict:
    """Our processed recordings -> {"X" [N,128,9], "y" [N] (0-5), "subject" [N] ("p1".."p5")}.

    Files follow the protocol name `<person>_<activity>_<take>.csv`, e.g. `p1_walking_01.csv`;
    the activity in the name is the label. A badly named or invalid file raises, naming the file,
    so a mislabelled recording can never slip into training.
    """
    import glob
    import os
    import re
    import pandas as pd
    from preprocess import to_windows
    from validation import validate_csv

    pat = re.compile(rf"^(p\d+)_({'|'.join(OWN_ACTS)})_\d+\.csv$")
    X, y, subj = [], [], []
    for path in sorted(glob.glob(os.path.join(root, "*.csv"))):
        name = os.path.basename(path)
        m = pat.match(name)
        if not m:
            raise ValueError(f"{name}: expected <person>_<activity>_<take>.csv with activity in {OWN_ACTS}")
        df = pd.read_csv(path)
        rep = validate_csv(df, cfg)
        if not rep["ok"]:
            raise ValueError(f"{name}: failed validation: {rep['issues']}")
        w = to_windows(df, cfg)
        X.append(w)
        y += [OWN_ACTS.index(m.group(2))] * len(w)
        subj += [m.group(1)] * len(w)
    if not X:
        raise FileNotFoundError(f"no CSVs in {root}")
    return {"X": np.concatenate(X), "y": np.array(y), "subject": np.array(subj)}

### `src/preprocess.py` · Nihal

In [4]:
show_file("src/preprocess.py")

`src/preprocess.py` · 79 lines

"""Owner: Nihal. Turn continuous phone recordings into UCI-style windows."""
import numpy as np
from scipy.signal import butter, filtfilt, medfilt

COLS = ["acc_x", "acc_y", "acc_z", "gyro_x", "gyro_y", "gyro_z"]


def _lowpass(x, cutoff, fs):
    b, a = butter(3, cutoff, btype="low", fs=fs)
    return filtfilt(b, a, x, axis=0)


def to_windows(df, cfg) -> np.ndarray:
    """Resample to 50 Hz, filter, cut 128-sample windows at cfg overlap. Returns [N,128,9].

    Input columns: timestamp (s), acc_x/y/z (g, incl. gravity), gyro_x/y/z (rad/s).
    Channel order matches data_loader.SIGNALS: body_acc, body_gyro, total_acc.
    """
    fs, win = cfg["sampling_rate_hz"], cfg["window_samples"]
    step = int(win * (1 - cfg["overlap"]))
    df = df.dropna(subset=["timestamp"] + COLS).sort_values("timestamp")
    df = df.drop_duplicates("timestamp")
    t = df["timestamp"].to_numpy(float)
    grid = np.arange(t[0], t[-1], 1 / fs)
    if len(grid) < win:
        return np.empty((0, win, 9))
    # ponytail: linear interp also fills short gaps; validate_csv rejects long ones
    x = np.stack([np.interp(grid, t, df[c].to_numpy(float)) for c in COLS], axis=1)
    x = medfilt(x, kernel_size=(3, 1))
    x = _lowpass(x, cfg["noise_cutoff_hz"], fs)
    total, gyro = x[:, :3], x[:, 3:]
    body = total - _lowpass(total, cfg["gravity_cutoff_hz"], fs)
    sig = np.concatenate([body, gyro, total], axis=1)          # [T, 9]
    starts = range(0, len(sig) - win + 1, step)
    return np.stack([sig[s:s + win] for s in starts])


def fit_norm(X, path):
    """Per-channel mean/std from TRAINING windows only; saved for the app."""
    mean, std = X.mean(axis=(0, 1)), X.std(axis=(0, 1))
    np.savez(path, mean=mean, std=std)
    return mean, std


def normalize(X, path):
    s = np.load(path)
    return (X - s["mean"]) / s["std"]


def gravity_angles(X) -> np.ndarray:
    """Angle (rad) between the gravity vector and each phone axis, per sample. [N,128,9] -> [N,128,3].

    Gravity = total_acc - body_acc (channels 6:9 minus 0:3). Sitting vs standing differ mostly in
    how the waist is tilted, so these angles are a direct posture cue.
    """
    g = X[..., 6:9] - X[..., 0:3]
    return np.arccos(np.clip(g / np.linalg.norm(g, axis=-1, keepdims=True), -1, 1))


if __name__ == "__main__":
    import pandas as pd
    cfg = {"sampling_rate_hz": 50, "window_samples": 128, "overlap": 0.5,
           "noise_cutoff_hz": 20, "gravity_cutoff_hz": 0.3}
    rng = np.random.default_rng(0)
    t = np.cumsum(rng.uniform(0.009, 0.011, 6000))          # ~100 Hz, jittery, 60 s
    df = pd.DataFrame({"timestamp": t, "acc_x": 1 + 0.2 * np.sin(2 * np.pi * 2 * t),
                       "acc_y": 0.0, "acc_z": 0.0, "gyro_x": 0.0, "gyro_y": 0.0,
                       "gyro_z": 0.1 * np.sin(2 * np.pi * t)})
    W = to_windows(df, cfg)
    n = (int((t[-1] - t[0]) * 50) - 128) // 64 + 1
    assert W.shape == (n, 128, 9), W.shape
    mid = W[2:-2]                                               # skip filter edges
    assert abs(mid[..., 6].mean() - 1) < 0.02                   # total acc keeps 1 g
    assert abs(mid[..., 0].mean()) < 0.02                       # body acc has gravity removed
    assert to_windows(df.iloc[:100], cfg).shape == (0, 128, 9)  # < 2.56 s -> no windows
    a = gravity_angles(W)
    assert a.shape == (n, 128, 3) and abs(a[2:-2, :, 0].mean()) < 0.05  # gravity along x -> angle 0
    print("ok", W.shape)

## Validation

### `src/validation.py` · Shravya

In [5]:
show_file("src/validation.py")

`src/validation.py` · 53 lines

"""Owner: Shravya."""
import numpy as np

from preprocess import COLS as SENSORS

COLS = ["timestamp"] + SENSORS
MAX_ACC_G, MAX_GYRO = 8.0, 35.0     # sane sensor limits (g, rad/s)
SHORT_GAP, LONG_GAP = 0.1, 1.0      # s: gaps > SHORT_GAP are counted, > LONG_GAP rejected
HZ_TOL = 0.10                       # allowed relative error vs cfg rate


def validate_csv(df, cfg) -> dict:
    """Checks columns, sampling rate, missing readings. Returns {ok, issues, est_hz, warnings}; short gaps are warnings only."""
    issues, warnings, est_hz = [], [], None
    missing = [c for c in COLS if c not in df.columns]
    if missing:
        return {"ok": False, "est_hz": None, "issues": [f"missing columns: {missing}"]}

    n_nan = int(df[COLS].isna().sum().sum())
    if n_nan:
        issues.append(f"{n_nan} NaN values")
    d = df.dropna(subset=COLS)
    t = d["timestamp"].to_numpy(float)
    if len(t) < 2 or not np.all(np.diff(t) > 0):
        return {"ok": False, "est_hz": None, "issues": issues + ["too few rows or timestamps not increasing"]}

    gaps = np.diff(t)
    est_hz = float(1 / np.median(gaps))
    want = cfg["sampling_rate_hz"]
    if est_hz < (1 - HZ_TOL) * want:  # faster is fine: to_windows resamples to 50 Hz
        issues.append(f"sampling rate {est_hz:.1f} Hz, need >= {want} Hz")
    if np.percentile(gaps, 95) > 2 * np.median(gaps):  # ponytail: crude jitter test
        issues.append("uneven sampling (high gap spread)")
    n_gaps = int((gaps > SHORT_GAP).sum())
    if n_gaps:
        warnings.append(f"{n_gaps} time gaps > {SHORT_GAP}s (largest {gaps.max():.2f}s)")
    if gaps.max() > LONG_GAP:
        issues.append(f"long gap > {LONG_GAP}s: reject file")

    need = cfg["window_samples"] / want
    if t[-1] - t[0] < need:
        issues.append(f"only {t[-1] - t[0]:.2f}s of data, need >= {need:.2f}s")

    acc, gyro = d[["acc_x", "acc_y", "acc_z"]].to_numpy(float), d[["gyro_x", "gyro_y", "gyro_z"]].to_numpy(float)
    if np.abs(acc).max() > MAX_ACC_G:
        issues.append(f"|acc| up to {np.abs(acc).max():.1f} > {MAX_ACC_G} g (units?)")
    if np.abs(gyro).max() > MAX_GYRO:
        issues.append(f"|gyro| up to {np.abs(gyro).max():.1f} > {MAX_GYRO} rad/s (deg/s?)")
    if abs(np.median(np.linalg.norm(acc, axis=1)) - 9.8) < 1.0:
        issues.append("median |acc| ~9.8: looks like m/s^2, divide by 9.80665")

    return {"ok": not issues, "est_hz": est_hz, "issues": issues, "warnings": warnings}

## Models, training, evaluation, phone export

### `src/models.py` · Yash

In [6]:
show_file("src/models.py")

`src/models.py` · 55 lines

"""Owner: Yash. Deep model uses Keras (tensorflow), not PyTorch."""
from sklearn.ensemble import RandomForestClassifier


def build_rf():
    return RandomForestClassifier(n_estimators=300, random_state=42, n_jobs=-1)


def build_cnn(input_shape=(128, 9), n_classes=6):
    from tensorflow import keras
    from tensorflow.keras import layers
    m = keras.Sequential([
        keras.Input(shape=input_shape),
        layers.Conv1D(64, 5, activation="relu"),
        layers.Conv1D(64, 5, activation="relu"),
        layers.Dropout(0.5),
        layers.MaxPooling1D(2),
        layers.Flatten(),
        layers.Dense(100, activation="relu"),
        layers.Dense(n_classes, activation="softmax"),
    ])
    m.compile(optimizer="adam", loss="sparse_categorical_crossentropy", metrics=["accuracy"])
    return m


def build_lstm(input_shape=(128, 9), n_classes=6):
    """Comparison model for the CNN: 2 stacked LSTMs on raw windows."""
    from tensorflow import keras
    from tensorflow.keras import layers
    m = keras.Sequential([
        keras.Input(shape=input_shape),
        layers.LSTM(64, return_sequences=True),
        layers.LSTM(64),
        layers.Dropout(0.5),
        layers.Dense(100, activation="relu"),
        layers.Dense(n_classes, activation="softmax"),
    ])
    m.compile(optimizer="adam", loss="sparse_categorical_crossentropy", metrics=["accuracy"])
    return m


def build_cnn_gravity(input_shape=(128, 9), n_angles=3, n_classes=6):
    """build_cnn plus a second input: window-mean gravity angles, concatenated before the dense layer."""
    from tensorflow import keras
    from tensorflow.keras import layers
    raw, ang = keras.Input(shape=input_shape), keras.Input(shape=(n_angles,))
    x = layers.Conv1D(64, 5, activation="relu")(raw)
    x = layers.Conv1D(64, 5, activation="relu")(x)
    x = layers.Dropout(0.5)(x)
    x = layers.Flatten()(layers.MaxPooling1D(2)(x))
    x = layers.Dense(100, activation="relu")(layers.Concatenate()([x, ang]))
    m = keras.Model([raw, ang], layers.Dense(n_classes, activation="softmax")(x))
    m.compile(optimizer="adam", loss="sparse_categorical_crossentropy", metrics=["accuracy"])
    return m

### `src/train.py` · Yash

In [7]:
show_file("src/train.py")

`src/train.py` · 28 lines

"""Owner: Yash."""


def fit(model, X_tr, y_tr, X_val, y_val, cfg):
    """Returns (model, history). history is None for sklearn models."""
    if not hasattr(model, "compile"):                     # sklearn: no epochs
        return model.fit(X_tr, y_tr), None
    from tensorflow import keras
    keras.utils.set_random_seed(cfg["seed"])
    stop = keras.callbacks.EarlyStopping(monitor="val_loss", patience=5, restore_best_weights=True)
    h = model.fit(X_tr, y_tr, validation_data=(X_val, y_val), epochs=30,
                  batch_size=64, callbacks=[stop], verbose=2)
    return model, h.history


def fine_tune(model, X_tr, y_tr, X_val, y_val, cfg, n_trainable=2):
    """Transfer learning: copy the UCI model, freeze all but the last `n_trainable` layers,
    recompile with Adam(1e-4) and train on our data. The original model is left untouched,
    so it can still be scored as the "no fine-tuning" baseline. Returns (model, history)."""
    from tensorflow import keras
    m = keras.models.clone_model(model)
    m.set_weights(model.get_weights())
    for layer in m.layers[:-n_trainable]:
        layer.trainable = False
    m.compile(optimizer=keras.optimizers.Adam(1e-4), loss="sparse_categorical_crossentropy",
              metrics=["accuracy"])
    return fit(m, X_tr, y_tr, X_val, y_val, cfg)

### `src/evaluate.py` · Yash

In [8]:
show_file("src/evaluate.py")

`src/evaluate.py` · 32 lines

"""Owner: Yash."""
import os
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
from sklearn.metrics import accuracy_score, f1_score, ConfusionMatrixDisplay

from inference import ACTS


def report(y_true, y_pred, out_dir, name="model") -> dict:
    """Accuracy, F1 per activity, confusion matrix PNG saved to out_dir/<name>_confusion.png."""
    os.makedirs(out_dir, exist_ok=True)
    f1 = f1_score(y_true, y_pred, average=None, labels=range(6))
    fig, ax = plt.subplots(figsize=(7, 6))
    ConfusionMatrixDisplay.from_predictions(y_true, y_pred, display_labels=ACTS, ax=ax,
                                            xticks_rotation=45, colorbar=False)
    ax.set_title(name); fig.tight_layout()
    fig.savefig(f"{out_dir}/{name}_confusion.png", dpi=120); plt.close(fig)
    return {"accuracy": accuracy_score(y_true, y_pred),
            "macro_f1": f1_score(y_true, y_pred, average="macro"),
            "f1_per_activity": dict(zip(ACTS, f1.round(4)))}


def plot_learning_curves(history, path):
    """Train vs validation loss and accuracy per epoch (the overfitting graph)."""
    fig, axs = plt.subplots(1, 2, figsize=(11, 4))
    for ax, k in zip(axs, ["loss", "accuracy"]):
        ax.plot(history[k], label="train"); ax.plot(history["val_" + k], label="validation")
        ax.set_xlabel("epoch"); ax.set_title(k); ax.legend()
    fig.tight_layout(); fig.savefig(path, dpi=120); plt.close(fig)

### `src/export.py` · Yash

In [9]:
show_file("src/export.py")

`src/export.py` · 60 lines

"""Export a Keras model to TFLite (float32 / dynamic-range / full int8) and benchmark it."""
import time
import numpy as np
import tensorflow as tf


def to_tflite(model, path, quant=None, rep_data=None):
    """quant: None (float32), "dynamic" (int8 weights) or "int8" (full int8, needs rep_data [N,128,9])."""
    conv = tf.lite.TFLiteConverter.from_keras_model(model)
    if quant in ("dynamic", "int8"):
        conv.optimizations = [tf.lite.Optimize.DEFAULT]
    if quant == "int8":
        assert rep_data is not None, "int8 needs rep_data"
        conv.representative_dataset = lambda: ([x[None].astype("float32")] for x in rep_data)
        conv.target_spec.supported_ops = [tf.lite.OpsSet.TFLITE_BUILTINS_INT8]
        conv.inference_input_type = conv.inference_output_type = tf.int8
    elif quant is not None and quant != "dynamic":
        raise ValueError(quant)
    blob = conv.convert()
    with open(path, "wb") as f:
        f.write(blob)
    return len(blob)


def predict_tflite(path, X, timed=False):
    """Run X [N,128,9] one window at a time. Returns probs [N,C] (and mean ms/window if timed)."""
    it = tf.lite.Interpreter(model_path=path, num_threads=1)
    it.allocate_tensors()
    i, o = it.get_input_details()[0], it.get_output_details()[0]
    out, t = [], 0.0
    for x in X:
        x = x[None].astype("float32")
        if i["dtype"] == np.int8:
            s, z = i["quantization"]
            x = np.clip(np.round(x / s + z), -128, 127).astype(np.int8)
        t0 = time.perf_counter()
        it.set_tensor(i["index"], x)
        it.invoke()
        y = it.get_tensor(o["index"])[0]
        t += time.perf_counter() - t0
        if o["dtype"] == np.int8:
            s, z = o["quantization"]
            y = (y.astype("float32") - z) * s
        out.append(y)
    out = np.array(out)
    return (out, 1000 * t / len(X)) if timed else out


if __name__ == "__main__":
    import os, tempfile
    m = tf.keras.Sequential([tf.keras.layers.Input((128, 9)), tf.keras.layers.Conv1D(4, 3),
                             tf.keras.layers.GlobalAveragePooling1D(), tf.keras.layers.Dense(6, activation="softmax")])
    X = np.random.randn(20, 128, 9).astype("float32")
    d = tempfile.mkdtemp()
    for q in (None, "dynamic", "int8"):
        p = os.path.join(d, f"{q}.tflite")
        to_tflite(m, p, q, X)
        assert predict_tflite(p, X[:3]).shape == (3, 6)
    print("ok")

## Prediction and live phone

### `src/inference.py` · Yashaswini

In [10]:
show_file("src/inference.py")

`src/inference.py` · 47 lines

"""Owner: Yashaswini."""
import numpy as np
import pandas as pd
from preprocess import to_windows, normalize

ACTS = ["WALKING", "WALKING_UPSTAIRS", "WALKING_DOWNSTAIRS", "SITTING", "STANDING", "LAYING"]


class DummyModel:
    """Always predicts STANDING. Used until a trained model file exists."""
    def predict(self, X, **_):
        p = np.zeros((len(X), 6)); p[:, 4] = 1
        return p


def predict_csv(path, model, cfg) -> pd.DataFrame:
    """One row per window: start_time, activity, confidence.

    `path` can be a file path or a file-like object (Streamlit upload).
    `model.predict(X)` must return class probabilities [N, 6].
    """
    df = pd.read_csv(path) if not isinstance(path, pd.DataFrame) else path
    X = normalize(to_windows(df, cfg), cfg["norm_stats"])
    if len(X) == 0:
        return pd.DataFrame(columns=["start_time", "activity", "confidence"])
    p = model.predict(X, verbose=0) if hasattr(model, "compile") else model.predict(X)
    step = cfg["window_samples"] * (1 - cfg["overlap"]) / cfg["sampling_rate_hz"]
    return pd.DataFrame({"start_time": df["timestamp"].min() + np.arange(len(X)) * step,
                         "activity": [ACTS[i] for i in p.argmax(1)],
                         "confidence": p.max(1).round(3)})


def smooth(pred_df, k=5) -> pd.DataFrame:
    """Centered rolling majority vote over `activity` (odd k, edges use a shorter window).

    Ties keep the original label if it is among the leaders. Returns a copy.
    """
    assert k % 2 == 1, "k must be odd"
    out, acts, h = pred_df.copy(), list(pred_df["activity"]), k // 2
    for i, orig in enumerate(acts):
        win = acts[max(0, i - h): i + h + 1]
        counts = {a: win.count(a) for a in set(win)}
        top = max(counts.values())
        if counts[orig] < top:
            out.iloc[i, out.columns.get_loc("activity")] = next(a for a in win if counts[a] == top)
    return out

### `src/live.py` · Yashaswini

In [11]:
show_file("src/live.py")

`src/live.py` · 165 lines

"""Live classification from the Phyphox app (Wi-Fi remote access) or a CSV replay.

Both sources return new samples as a DataFrame in the app's input format:
timestamp (s), acc_x..z (g, incl. gravity), gyro_x..z (rad/s).
"""
import json
import time
import urllib.request

import numpy as np
import pandas as pd

from inference import ACTS
from preprocess import COLS, normalize, to_windows

G = 9.80665
ACC = ["accX", "accY", "accZ"]          # buffer names in docs/har_live.phyphox
GYR = ["gyrX", "gyrY", "gyrZ"]
# Phone axis (index, sign) feeding UCI x, y, z, applied to acc and gyro alike.
# UCI x is vertical; a phone upright at the waist has its own y vertical, so swap x/y
# and flip z to stay right-handed. ponytail: untested on a real phone; check with the
# standing test in docs/live_demo.md and edit here if acc_x is not ~ +1 g.
AXES = [(1, 1), (0, 1), (2, -1)]


def _cols(buf, names):
    """[n, len(names)] array; buffers caught mid-update are cut to the shortest, NaN rows dropped."""
    cols = [np.asarray(buf[b]["buffer"], float) for b in names]
    n = min(len(c) for c in cols)
    x = np.stack([c[:n] for c in cols], 1)
    return x[~np.isnan(x).any(1)]


class PhyphoxSource:
    """Polls http://<ip>:8080/get for samples newer than the last ones seen."""

    def __init__(self, address, timeout=2.0):
        self.base = "http://" + address.strip().removeprefix("http://").rstrip("/")
        if ":" not in self.base[7:]:
            self.base += ":8080"
        self.timeout = timeout
        self.t_acc = self.t_gyr = -1.0
        self.last_gyr = None  # last gyro sample, keeps interpolation continuous across polls

    def _call(self, path):
        with urllib.request.urlopen(self.base + path, timeout=self.timeout) as r:
            return json.loads(r.read())

    def start(self):
        # No cmd=clear: on a real phone it shut down phyphox's web server. Skip old samples by
        # starting the thresholds at the newest timestamps already in the buffers.
        buf = self._call("/get?acc_time&gyr_time")["buffer"]
        last = lambda b: (buf.get(b, {}).get("buffer") or [None])[-1]
        self.t_acc = last("acc_time") if last("acc_time") is not None else -1.0
        self.t_gyr = last("gyr_time") if last("gyr_time") is not None else -1.0
        self._call("/control?cmd=start")

    def stop(self):
        self._call("/control?cmd=stop")

    def read(self):
        ta, tg = f"{self.t_acc:.6f}", f"{self.t_gyr:.6f}"
        q = "&".join([f"{b}={ta}|acc_time" for b in ACC] + [f"acc_time={ta}"]
                     + [f"{b}={tg}|gyr_time" for b in GYR] + [f"gyr_time={tg}"])
        buf = self._call("/get?" + q.replace("|", "%7C"))["buffer"]  # phyphox answers 400 to a raw |
        if "gyr_time" not in buf or "acc_time" not in buf:
            raise RuntimeError('wrong experiment open in phyphox. Open "HAR live (acc + gyro)", '
                               "which records the accelerometer and gyroscope together.")
        a, g = _cols(buf, ["acc_time"] + ACC), _cols(buf, ["gyr_time"] + GYR)
        if len(g):
            self.t_gyr = g[-1, 0]
            g = g if self.last_gyr is None else np.vstack([self.last_gyr, g])
            self.last_gyr = g[-1:]
        elif self.last_gyr is not None:
            g = self.last_gyr
        if not len(a) or not len(g):
            return pd.DataFrame(columns=["timestamp"] + COLS)
        self.t_acc = a[-1, 0]
        # gyro has its own clock; resample it onto the accelerometer timestamps
        gi = np.stack([np.interp(a[:, 0], g[:, 0], g[:, i]) for i in (1, 2, 3)], 1)
        acc = np.stack([s * a[:, 1 + i] / G for i, s in AXES], 1)
        gyr = np.stack([s * gi[:, i] for i, s in AXES], 1)
        return pd.DataFrame(np.hstack([a[:, :1], acc, gyr]), columns=["timestamp"] + COLS)


class ReplaySource:
    """Plays a CSV back at real time, looping, so the page works without a phone."""

    def __init__(self, path, speed=1.0):
        self.df = pd.read_csv(path).sort_values("timest

## Camera pipeline

### `src/video_pose.py` · Yyug

In [12]:
show_file("src/video_pose.py")

`src/video_pose.py` · 319 lines

"""Camera pipeline: video -> MediaPipe Pose keypoints -> normalised windows for a Keras model."""
import argparse, csv, os, re, sys, time
import numpy as np
import yaml

sys.path.insert(0, os.path.dirname(__file__))

ROOT = os.path.join(os.path.dirname(__file__), "..")


def _cfg():
    with open(os.path.join(ROOT, "config", "config.yaml")) as f:
        return yaml.safe_load(f)


def extract_keypoints(video_path, fps=25) -> np.ndarray:
    """[frames, 33, 4] (x, y, z, visibility), resampled to `fps`; NaN rows where no person is found."""
    import cv2, mediapipe as mp
    from mediapipe.tasks.python import BaseOptions, vision
    model = os.path.join(ROOT, _cfg()["video"]["pose_model"])
    opts = vision.PoseLandmarkerOptions(base_options=BaseOptions(model_asset_path=model),
                                        running_mode=vision.RunningMode.VIDEO)
    cap = cv2.VideoCapture(video_path)
    if not cap.isOpened():
        raise IOError(f"cannot open {video_path}")
    src_fps = cap.get(cv2.CAP_PROP_FPS)
    n_src = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
    n_out = int(n_src / src_fps * fps)
    want = {min(round(k * src_fps / fps), n_src - 1): k for k in range(n_out)}  # source frame -> output index
    out = np.full((n_out, 33, 4), np.nan, np.float32)
    with vision.PoseLandmarker.create_from_options(opts) as lm:
        for i in range(n_src):
            if i not in want:
                cap.grab()
                continue
            ok, bgr = cap.read()
            if not ok:
                break
            img = mp.Image(image_format=mp.ImageFormat.SRGB, data=cv2.cvtColor(bgr, cv2.COLOR_BGR2RGB))
            res = lm.detect_for_video(img, int(want[i] * 1000 / fps))
            if res.pose_landmarks:
                out[want[i]] = [[p.x, p.y, p.z, p.visibility] for p in res.pose_landmarks[0]]
    cap.release()
    return out


def normalize_pose(kp, max_gap=5) -> np.ndarray:
    """[frames, 33, 2]: x,y centred on hip midpoint, scaled by torso length; NaN gaps <= max_gap interpolated."""
    xy = np.asarray(kp, np.float64)[:, :, :2]
    hip = xy[:, [23, 24]].mean(1)
    torso = np.linalg.norm(xy[:, [11, 12]].mean(1) - hip, axis=1)
    torso[torso == 0] = np.nan
    out = (xy - hip[:, None]) / torso[:, None, None]
    bad = np.isnan(out).any((1, 2))
    good = np.flatnonzero(~bad)
    if len(good) == 0:
        return out
    flat = out.reshape(len(out), -1)
    t = np.arange(len(out))
    for c in range(flat.shape[1]):
        flat[bad, c] = np.interp(t[bad], good, flat[good, c], left=np.nan, right=np.nan)
    # re-blank gaps longer than max_gap
    i = 0
    while i < len(bad):
        if bad[i]:
            j = i
            while j < len(bad) and bad[j]:
                j += 1
            if j - i > max_gap:
                flat[i:j] = np.nan
            i = j
        else:
            i += 1
    return flat.reshape(out.shape)


def to_windows(kp_norm, window, overlap) -> np.ndarray:
    """[N, window, 66]; windows containing NaN are dropped."""
    flat = np.asarray(kp_norm).reshape(len(kp_norm), -1)
    step = max(1, int(window * (1 - overlap)))
    w = [flat[s:s + window] for s in range(0, len(flat) - window + 1, step)]
    w = [x for x in w if not np.isnan(x).any()]
    return np.stack(w) if w else np.empty((0, window, flat.shape[1]))


def load_video(root, cfg) -> dict:
    """Read cached <person>_<activity>_<take>.npz keypoints from root -> {"X", "y", "subject"}."""
    v = cfg["video"]
    classes = v["classes"]
    pat = re.compile(rf"^([^_]+)_({'|'.join(classes)})_\d+\.npz$")
    X, y, subj = [], [], []
    for name in sorted(os.listdir(root)):
        if not name.endswith(".npz"):
            continue
        m = pat.match(name)
        if not m:
            raise ValueError(f"{name}: expected <person>_<activity>_<take>.npz with activity in {classes}")
        kp = np.load(os.path.join(root, name))["kp"]
        w = to_windows(normalize_pose(kp, v["max_gap"]), v["window_frames"], v["overlap"])
        

### `notebooks/09_stairs_direction.py` · Yyug

In [13]:
show_file("notebooks/09_stairs_direction.py")

`notebooks/09_stairs_direction.py` · 65 lines

"""Stairs direction (up vs down) for the camera model. Run from the repo root:
    python notebooks/09_stairs_direction.py

Two stages: the existing 3-class model (notebook 08) says walking / standing / stairs; a second
Random Forest, same settings, says up or down for windows called stairs. Stage 1 is not retrained.

Direction labels were set by eye from video frames (not from any model output). "front/back" clips
were filmed from the top of the flight, so up = walking toward the camera: a model could learn
"toward the camera" instead of direction, so side-view clips are scored separately as the real test.
A (3) is too short; A (5) and D (6) are side views whose direction could not be told: left out.
"""
import glob, os, sys
import joblib, numpy as np, yaml
from sklearn.ensemble import RandomForestClassifier

sys.path.insert(0, "src")
from video_pose import MODEL, window_features

# keypoint file -> [(direction, start s, end s, view)]; video file in the comment
SEG = {"p1_stairs_05": [("upstairs", 0, 9.0, "front/back"), ("downstairs", 10.5, 20.0, "front/back")],  # A (8)
       "p3_stairs_01": [("upstairs", 0, 9.5, "front/back"), ("downstairs", 10.5, 19.0, "front/back")],  # C (5)
       "p4_stairs_02": [("upstairs", 0, 6.0, "front/back"), ("downstairs", 6.5, 12.56, "front/back")],  # D (7)
       "p2_stairs_02": [("downstairs", 0, 4.6, "front/back")],                                          # B (5)
       "p2_stairs_03": [("upstairs", 0, 4.0, "front/back")],                                            # B (7)
       "p1_stairs_03": [("upstairs", 0, 14.76, "side")],                                                # A (4)
       "p2_stairs_01": [("upstairs", 0, 10.52, "side")],                                                # B (4)
       "p1_stairs_01": [("downstairs", 0, 14.44, "side")]}                                              # A (1)
DIRS = ["upstairs", "downstairs"]
RF = lambda: RandomForestClassifier(300, min_samples_leaf=3, class_weight="balanced", random_state=42, n_jobs=-1)

cfg = yaml.safe_load(open("config/config.yaml"))
fps = cfg["video"]["fps"]
bundle = joblib.load(MODEL)   # use the same features as the saved stage-1 model (notebook 10 or 08)
X, y, sub, view = [], [], [], []
for name, segs in SEG.items():
    kp = np.load(f"data/video/keypoints/{name}.npz")["kp"]
    for d, s, e, vw in segs:
        w = window_features(kp[int(s * fps):int(e * fps)], bundle, cfg)[1]
        if w is None:
            continue
        X.append(w); y += [DIRS.index(d)] * len(w); sub += [name[:2]] * len(w); view += [vw] * len(w)
F, y, sub, view = np.concatenate(X), np.array(y), np.array(sub), np.array(view)
print(f"{len(y)} stairs windows: {(y == 0).sum()} up, {(y == 1).sum()} down")

# leave one person out: direction accuracy on the held-out person's true stairs windows
right = np.zeros(len(y), bool)
for p in sorted(set(sub)):
    te = sub == p
    right[te] = RF().fit(F[~te], y[~te]).predict(F[te]) == y[te]
    print(f"{p}: {right[te].mean():.2f} of {te.sum()} windows")
for vw in ("front/back", "side"):
    print(f"{vw:10s}: {right[view == vw].mean():.2f} of {(view == vw).sum()} windows")
rng, null = np.random.default_rng(0), []
for _ in range(10):  # same test with training labels shuffled: what chance looks like here
    r = np.zeros(len(y), bool)
    for p in sorted(set(sub)):
        te = sub == p
        r[te] = RF().fit(F[~te], rng.permutation(y[~te])).predict(F[te]) == y[te]
    null.append(r.mean())
print(f"all: {right.mean():.2f} (training labels shuffled: {np.mean(null):.2f} ± {np.std(null):.2f})")

bundle["direction"] = {"model": RF().fit(F, y), "classes": DIRS}
joblib.dump(bundle, MODEL)
print(f"added the direction model to {MODEL}")

### `notebooks/10_body_points.py` · Yyug

In [14]:
show_file("notebooks/10_body_points.py")

`notebooks/10_body_points.py` · 107 lines

"""Camera model on better use of the body points. Run from the repo root:
    python notebooks/10_body_points.py

Candidates, fixed before any test result:
  F0  the 23 features of notebook 08 (today's model)
  F1  the same 23 on undistorted points (MediaPipe x is in widths, y in heights; videos are portrait)
  F2  F1 + 15 features in a 3D body frame (x, y, depth z; torso = vertical axis): knee and hip angles,
      foot and step height, so they do not depend on the camera angle
  F3  F2 + 4 whole-body movement features (travel and approach speed through the frame)
each with a Random Forest (notebook 08 settings) or gradient boosting.

Honest score: nested leave-one-person-out. For each test person, the candidate is chosen using only
the other people, then scored once on the test person. p5 (one clip) is training-only.
"""
import glob, os, sys
import joblib, numpy as np, yaml
from sklearn.ensemble import HistGradientBoostingClassifier, RandomForestClassifier
from sklearn.metrics import accuracy_score, f1_score

sys.path.insert(0, "src")
from video_pose import MODEL, TRAIN_SIZE, window_features

cfg = yaml.safe_load(open("config/config.yaml"))
C3, DIRS = cfg["video"]["classes"], ["upstairs", "downstairs"]
src = open("notebooks/09_stairs_direction.py", encoding="utf-8").read()
SEG = eval(src[src.index("SEG = ") + 6: src.index("DIRS = ")].strip())   # direction labels, set by eye
FPS = cfg["video"]["fps"]
SETS = {"F0": None, "F1": 23, "F2": 38, "F3": 42}
MODELS = {"RF": lambda: RandomForestClassifier(300, min_samples_leaf=3, class_weight="balanced", random_state=42, n_jobs=-1),
          "HGB": lambda: HistGradientBoostingClassifier(max_iter=200, learning_rate=0.05, class_weight="balanced", random_state=42)}


def feats(kp, cols):
    return window_features(kp, {"cols": cols}, cfg, TRAIN_SIZE)[1]


# stage 1 data: every window of every clip, 3 classes
F = {k: [] for k in SETS}; y, who = [], []
for f in sorted(glob.glob("data/video/keypoints/*.npz")):
    p, act, _ = os.path.basename(f)[:-4].split("_")
    kp = np.load(f)["kp"]
    for k, c in SETS.items():
        X = feats(kp, c)
        if X is not None: F[k].append(X)
    n = len(window_features(kp, {}, cfg)[0])
    assert all(len(window_features(kp, {"cols": c}, cfg)[0]) == n for c in (23,)), "window sets differ"
    y += [C3.index(act)] * n; who += [p] * n
F = {k: np.concatenate(v) for k, v in F.items()}; y, who = np.array(y), np.array(who)
TEST = ["p1", "p2", "p3", "p4"]
print(f"{len(y)} windows: {dict(zip(C3, np.bincount(y)))}")


def lopo(X, mk, people, mask):
    P = np.full(len(y), -1)
    for t in people:
        tr, te = mask & (who != t), mask & (who == t)
        P[te] = mk().fit(X[tr], y[tr]).predict(X[te])
    return P


def score(P, people):
    a = [accuracy_score(y[who == t], P[who == t]) for t in people]
    f = [f1_score(y[who == t], P[who == t], average="macro", labels=range(3)) for t in people]
    return np.mean(a), np.std(a), np.mean(f), a


print("\nplain leave-one-person-out (choosing the best of these afterwards would be optimistic):")
plain = {}
for k in SETS:
    for m, mk in MODELS.items():
        acc, sd, f1, per = score(lopo(F[k], mk, TEST, np.ones(len(y), bool)), TEST)
        plain[(k, m)] = f1
        print(f"  {k} {m:3s}: accuracy {acc:.2f} ± {sd:.2f}, macro F1 {f1:.2f}  per person {np.round(per, 2).tolist()}")

print("\nnested (honest):")
outer = []
for t in TEST:
    inner, m = [q for q in TEST if q != t], who != t
    best = max(((k, mm) for k in SETS for mm in MODELS),
               key=lambda km: score(lopo(F[km[0]], MODELS[km[1]], inner, m), inner)[2])
    te = who == t
    z = MODELS[best[1]]().fit(F[best[0]][m], y[m]).predict(F[best[0]][te])
    outer.append((accuracy_score(y[te], z), f1_score(y[te], z, average="macro", labels=range(3))))
    print(f"  test {t}: chose {best[0]} {best[1]} -> accuracy {outer[-1][0]:.2f}, macro F1 {outer[-1][1]:.2f}")
print(f"  NESTED: accuracy {np.mean([o[0] for o in outer]):.2f} ± {np.std([o[0] for o 

## Web app (Streamlit)

### `app/streamlit_app.py` · Yashaswini

In [15]:
show_file("app/streamlit_app.py")

`app/streamlit_app.py` · 17 lines

"""Entry point. Run from the repo root: streamlit run app/streamlit_app.py

Top navigation over four pages; each page file draws its own content.
"""
import streamlit as st

st.set_page_config(page_title="Activity Recognition", page_icon=":material/directions_run:", layout="wide")
st.navigation(
    [
        st.Page("overview.py", title="Overview", icon=":material/home:", default=True),
        st.Page("home.py", title="Phone recording", icon=":material/upload_file:", url_path="Phone"),
        st.Page("pages/1_Live.py", title="Live", icon=":material/sensors:", url_path="Live"),
        st.Page("pages/2_Video.py", title="Video", icon=":material/videocam:", url_path="Video"),
    ],
    position="top",
).run()

### `app/overview.py` · Yashaswini

In [16]:
show_file("app/overview.py")

`app/overview.py` · 130 lines

"""Overview: what the project does, with the real sample recording as the centrepiece.

Presentation only. Motion is CSS (see theme.css) and respects prefers-reduced-motion.
"""
import html
import os
import pathlib
import sys

import numpy as np
import pandas as pd
import streamlit as st

sys.path.insert(0, os.path.dirname(__file__))
from components.ui import PHONE_ACCURACY, act, footer, icon, kinetic, load_css, section, tape_html

st.set_page_config(page_title="Overview · Activity Recognition", page_icon=":material/home:", layout="wide")
load_css()

SAMPLE = pathlib.Path(__file__).resolve().parent / "demo" / "uci_test_person_12.csv"
# Seconds per activity in the sample, as the phone model labels it (see the Phone recording page).
SEGMENTS = [("WALKING", 21), ("WALKING_DOWNSTAIRS", 19), ("WALKING_UPSTAIRS", 17),
            ("SITTING", 21), ("STANDING", 21), ("LAYING", 19)]
CAPTIONS = {
    "WALKING": "A steady beat, about two steps a second.",
    "WALKING_UPSTAIRS": "The same stride, lifting on every step.",
    "WALKING_DOWNSTAIRS": "A firmer landing on every step.",
    "SITTING": "Still, with the phone tipped back a little.",
    "STANDING": "Still, with the phone upright.",
    "LAYING": "Still, with gravity along another axis.",
}


def _md(markup: str) -> None:
    st.markdown(markup, unsafe_allow_html=True)


@st.cache_data(show_spinner=False)
def load_sample(points: int = 1200, width: int = 1200, height: int = 176) -> tuple[str, float] | None:
    """(SVG path of the sample's acceleration magnitude, length in seconds). 1 g sits mid-height.
    None if the file is missing."""
    try:
        df = pd.read_csv(SAMPLE, usecols=["timestamp", "acc_x", "acc_y", "acc_z"])
    except (OSError, ValueError):
        return None
    mag = np.linalg.norm(df[["acc_x", "acc_y", "acc_z"]].to_numpy(), axis=1)
    picks = np.linspace(0, len(mag) - 1, points).astype(int)
    y = np.clip(0.5 - (mag[picks] - 1.0) / 3.0, 0.02, 0.98) * height
    x = np.linspace(0, width, points)
    path = "M" + " L".join(f"{a:.1f},{b:.1f}" for a, b in zip(x, y))
    return path, float(df["timestamp"].iloc[-1] - df["timestamp"].iloc[0])


def reel() -> str:
    slides = "".join(
        f'<div class="slide {act(k).cls}" style="--i:{i}" aria-hidden="true">{icon(act(k).icon)}'
        f'<div><span class="word"><span>{html.escape(act(k).short)}</span></span>'
        f'<p class="cap">{html.escape(CAPTIONS[k])}</p></div></div>'
        for i, k in enumerate(CAPTIONS))
    names = ", ".join(act(k).label.lower() for k in CAPTIONS)
    return f'<div class="reel" role="img" aria-label="The six activities the model recognises: {names}">{slides}</div>'


def scan_figure() -> str:
    sample = load_sample()
    if sample is None:
        return ""
    path, seconds = sample
    bands = "".join(f'<i class="{act(k).cls}" style="--w:{n}"></i>' for k, n in SEGMENTS)
    svg = ('<svg viewBox="0 0 1200 176" preserveAspectRatio="none" role="img" '
           'aria-label="Acceleration magnitude of the sample recording over two minutes">'
           f'<path class="scan-trace" d="{path}"/></svg>')
    return (f'<figure class="scan reveal" style="--dur:14s"><div class="scan-plot">'
            f'<div class="scan-sig"><div class="scan-bands" aria-hidden="true">{bands}</div>{svg}'
            f'<div class="scan-head" aria-hidden="true"></div></div>'
            f'{tape_html(SEGMENTS, step=1.0, total=seconds)}</div>'
            f'<figcaption>The line is the phone\'s acceleration, straight from the sample recording. '
            f'The colours are what the model made of it, 2.56 seconds at a time.</figcaption></figure>')


# ---------------------------------------------------------------------------
# Hero: kinetic headline on the left, the six activities wiping past on the right
# ---------------------------------------------------------------------------
copy, visual = st.columns([1.15, 1], gap="large", vertical_alignment="center")
with copy:
    headline_text = "Your phone ca

### `app/home.py` · Yashaswini

In [17]:
show_file("app/home.py")

`app/home.py` · 221 lines

"""Owner: Yashaswini. Phone recording page; app/streamlit_app.py routes to it.

Controls on the left, the result on the right. Summary tab: what happened, in order.
Details tab: confidence, per-window predictions, download.

Presentation only: all data and inference logic is unchanged (see DESIGN.md).
"""
import os
import sys

import pandas as pd
import streamlit as st
import yaml

sys.path.insert(0, "src")
from inference import ACTS, DummyModel, predict_csv, smooth
from validation import validate_csv

sys.path.insert(0, os.path.join(os.path.dirname(__file__)))
from components.ui import (
    act,
    coalesce,
    confidence_chart,
    empty_state,
    facts,
    footer,
    headline,
    ledger,
    listing,
    load_css,
    mmss,
    notice,
    page_header,
    rail_mark,
    section,
    summarize,
    tape,
)

MODEL = "models/cnn_uci.keras"
DEMO = "app/demo/uci_test_person_12.csv"  # real UCI HAR test person, never trained on

st.set_page_config(page_title="Phone recording · Activity Recognition", page_icon=":material/upload_file:", layout="wide")
load_css()


@st.cache_resource(show_spinner="Loading the model…")
def load_model():
    if os.path.exists(MODEL):
        from tensorflow import keras
        return keras.models.load_model(MODEL), "1D-CNN (UCI HAR)"
    return DummyModel(), "dummy (always Standing)"


# ---------------------------------------------------------------------------
# Header and controls (left rail)
# ---------------------------------------------------------------------------
cfg = yaml.safe_load(open("config/config.yaml"))
page_header(
    "Phone recording",
    "Upload a recording from a phone worn at the waist. The model labels each 2.56-second window "
    "as walking, stairs, sitting, standing or lying.",
)
rail, canvas = st.columns([1, 2.4], gap="large")

with rail:
    rail_mark()
    up = st.file_uploader("Phone recording (CSV)", type="csv")
    if up is not None:
        st.session_state.demo = False
    elif not st.session_state.get("demo"):
        if st.button("Try the sample recording", type="primary", icon=":material/play_arrow:", width="stretch"):
            st.session_state.demo = True
            st.rerun()
        st.caption("Two minutes of a real person from the UCI HAR test set, whom the model never trained on.")
    else:
        notice("Showing the sample: UCI HAR test person 12, not used in training.")
        if st.button("Clear sample", width="stretch"):
            st.session_state.demo = False
            st.rerun()
    source = up if up is not None else (DEMO if st.session_state.get("demo") else None)

    with st.expander("Analysis settings"):
        smooth_on = st.checkbox(
            "Smooth predictions",
            value=True,
            help=f"Majority vote over {cfg['smooth_window']} windows removes single-window flicker.",
        )
        cfg["window_samples"] = st.slider(
            "Window length (samples)", min_value=32, max_value=512,
            value=int(cfg["window_samples"]), step=32,
            help="128 samples = 2.56 s at 50 Hz, what the model was trained on.",
        )
        step_fraction = st.slider(
            "Step size (fraction of window)", min_value=0.1, max_value=1.0,
            value=float(1 - cfg["overlap"]), step=0.05,
            help="0.5 = windows overlap by half. Smaller steps give more, overlapping windows.",
        )
        cfg["overlap"] = 1 - step_fraction
        if cfg["window_samples"] != 128:
            notice("The CNN was trained on 128-sample windows; other lengths may reduce prediction quality.", "warn")

# ---------------------------------------------------------------------------
# Result (right canvas)
# ---------------------------------------------------------------------------
with canvas:
    if source is None:
        empty_state(
            "No recording yet",
            "Upload a CSV from a phone worn at the waist, or try the sample recording. "
            "Columns: timestamp (s), acc_x/y/z (g), g

### `app/pages/1_Live.py` · Yashaswini

In [18]:
show_file("app/pages/1_Live.py")

`app/pages/1_Live.py` · 175 lines

"""Live activity from the Phyphox app over Wi-Fi, or a replay of the demo recording (app/demo).
Phone setup: docs/live_demo.md. Run from the repo root: streamlit run app/streamlit_app.py

Controls on the left, the status board on the right. Presentation only: the polling and
prediction logic is unchanged (see DESIGN.md).
"""
import os
import sys
from pathlib import Path

import streamlit as st
import yaml

sys.path.insert(0, "src")
from inference import DummyModel
from live import LiveClassifier, PhyphoxSource, ReplaySource

sys.path.insert(0, os.path.join(os.path.dirname(__file__), ".."))
from components.ui import (
    board,
    coalesce,
    footer,
    load_css,
    notice,
    page_header,
    rail_mark,
    section,
    setup_steps,
    tape,
)

MODEL = "models/cnn_uci.keras"
# Real UCI HAR recording of test person 12 (never trained on): walking, down, up, sit, stand, lie
SAMPLE = Path(__file__).resolve().parents[1] / "demo" / "uci_test_person_12.csv"
HISTORY = 30  # predictions kept in the strip (one per second)
PHONE, REPLAY = "Phone", "Demo replay"

st.set_page_config(page_title="Live · Activity Recognition", page_icon=":material/sensors:", layout="wide")
load_css()


@st.cache_resource(show_spinner="Loading the model…")
def load_model():
    if os.path.exists(MODEL):
        from tensorflow import keras
        return keras.models.load_model(MODEL)
    return DummyModel()


cfg = yaml.safe_load(open("config/config.yaml"))
ss = st.session_state
ss.setdefault("live", None)
ss.setdefault("history", [])
ss.setdefault("error", None)

page_header(
    "Live",
    "Wear your phone at the waist and see what you are doing, updated every second. "
    "No phone? Play the demo recording.",
)
rail, canvas = st.columns([1, 2.4], gap="large")

# ---------------------------------------------------------------------------
# Controls (left rail)
# ---------------------------------------------------------------------------
with rail:
    rail_mark()
    mode = st.segmented_control("Source", [PHONE, REPLAY], default=PHONE, required=True,
                                disabled=ss.live is not None)
    ss["_live_mode"] = mode  # the status chip reads it

    if mode == PHONE:
        ip = st.text_input("Phone address shown in Phyphox", key="phone_ip", placeholder="192.168.0.42:8080",
                           disabled=ss.live is not None)
    else:
        ip = ss.get("phone_ip", "")
        ss.phone_ip = ip  # keep what was typed while the field is hidden

    sample_available = SAMPLE.is_file()
    if mode == REPLAY and not sample_available:
        notice(
            "Replay is unavailable because app/demo/uci_test_person_12.csv is missing. "
            "Restore it from git, or choose Phone.",
            "warn",
        )

    if ss.live is None:
        if st.button("Start", type="primary", width="stretch", icon=":material/play_arrow:",
                     disabled=mode == REPLAY and not sample_available):
            try:
                src = PhyphoxSource(ip) if mode == PHONE else ReplaySource(SAMPLE)
                src.start()
                ss.live = (src, LiveClassifier(load_model(), cfg))
                ss.history = []
                ss.error = None
            except FileNotFoundError:
                if mode == REPLAY:
                    ss.error = (
                        "The replay sample became unavailable. Restore "
                        "app/demo/uci_test_person_12.csv from git, or choose Phone."
                    )
                else:
                    ss.error = "Could not start: the requested file was not found."
            except Exception as e:
                ss.error = f"Could not start: {e}"
            st.rerun()
    else:
        if st.button("Stop", width="stretch", icon=":material/stop:"):
            try:
                ss.live[0].stop()
            except Exception:
                pass
            ss.live = None
            st.rerun()

    if mode == PHONE and ss.live is None:
   

### `app/pages/2_Video.py` · Yashaswini

In [19]:
show_file("app/pages/2_Video.py")

`app/pages/2_Video.py` · 271 lines

"""Activity from a phone video: MediaPipe body keypoints -> hand-built features -> Random Forest.
Experimental: 3 classes, trained on 5 people. Run from the repo root: streamlit run app/streamlit_app.py

The video and its controls on the left, the result on the right. Presentation only: pose
estimation and prediction logic are unchanged (see DESIGN.md).
"""
import os
import sys
import tempfile

import streamlit as st
import yaml

sys.path.insert(0, "src")
from inference import smooth
from video_pose import MODEL, predict_live, predict_video

sys.path.insert(0, os.path.join(os.path.dirname(__file__), ".."))
from components.ui import (
    act,
    board,
    coalesce,
    empty_state,
    facts,
    footer,
    headline,
    ledger,
    load_css,
    mmss,
    page_header,
    rail_mark,
    section,
    summarize,
    tape,
)

# Classes of the camera model. "stairs" only appears with a model saved before notebook 09.
VIDEO_CLASSES = ["walking", "standing", "upstairs", "downstairs"]
DOT_FILL, DOT_RING = (255, 255, 255), (20, 33, 61)  # pose points drawn on the webcam frame (RGB)

st.set_page_config(page_title="Video · Activity Recognition", page_icon=":material/videocam:", layout="wide")
load_css()

VIDEO_NOTE = ("Camera model: on a person it never saw, 64% of windows right for walking / standing / stairs "
              "(macro F1 0.65); stairs direction right 85% of the time.")
page_header(
    "Activity from video",
    "Upload a video of one person, or use your webcam, to see when they walk, stand, "
    "or walk up or down stairs.",
)

# ---------------------------------------------------------------------------
# Model availability check
# ---------------------------------------------------------------------------
if not os.path.exists(MODEL):
    st.error(
        "No video model yet. Run notebooks/08_video_features.ipynb "
        "to create models/video_rf.joblib."
    )
    st.stop()


@st.cache_resource(show_spinner="Loading the model…")
def load_bundle():
    import joblib
    return joblib.load(MODEL)


cfg = yaml.safe_load(open("config/config.yaml"))


def run_live(bundle, cfg, canvas):
    """Webcam -> pose per frame -> label from the last 2 s, refreshed every 0.5 s.
    Needs the app to run on the computer the camera is plugged into.
    Controls render where this is called (the rail); the label and frame go to `canvas`."""
    import time
    import cv2
    import mediapipe as mp
    import numpy as np
    from mediapipe.tasks.python import BaseOptions, vision

    cam = st.number_input("Camera number", 0, 5, 0, help="0 is usually the built-in webcam.")
    c1, c2 = st.columns(2)
    if c1.button("Start", type="primary", width="stretch"):
        st.session_state.cam_on = True
    if c2.button("Stop", width="stretch"):
        st.session_state.cam_on = False

    with canvas:
        if not st.session_state.get("cam_on"):
            empty_state(
                "Camera off",
                "Stand 2 to 3 m from the camera with your whole body in frame, then press Start. "
                "The label appears after 2 s and updates twice a second.",
            )
            return

        label, frame_box = st.empty(), st.empty()
        cap = cv2.VideoCapture(int(cam))
        if not cap.isOpened():
            st.session_state.cam_on = False
            st.error(f"Can't open camera {cam}. Try another number, or allow camera access for the terminal.")
            return
        opts = vision.PoseLandmarkerOptions(
            base_options=BaseOptions(model_asset_path=cfg["video"]["pose_model"]),
            running_mode=vision.RunningMode.VIDEO,
        )
        times, kps, t0, last, ms = [], [], time.time(), -1.0, -1
        try:  # Stop reruns the script, which ends this loop; finally frees the camera
            with vision.PoseLandmarker.create_from_options(opts) as lm:
                while st.session_state.get("cam_on"):
                    ok, bgr = cap.read()
                    if not ok:
            

### `app/components/ui.py` · Yashaswini

In [20]:
show_file("app/components/ui.py")

`app/components/ui.py` · 272 lines

"""
app/components/ui.py
Shared presentation helpers. No ML logic.

Colours live in styles/theme.css as CSS variables; this module only emits class names and
structure. Every helper renders one HTML block through st.markdown(unsafe_allow_html=True),
because st.html strips inline <svg>. Keep the strings on one line: a blank line ends the
Markdown HTML block and the rest renders as a code block.
"""
from __future__ import annotations

import html
import itertools
import pathlib
from collections.abc import Iterable, Mapping, Sequence
from dataclasses import dataclass

import streamlit as st

_CSS_PATH = pathlib.Path(__file__).resolve().parent.parent / "styles" / "theme.css"
esc = html.escape

PHONE_ACCURACY = 91  # % of windows right on the 9 never-seen UCI HAR people (reports/results.md)
PHONE_NOTE = (f"Phone model: {PHONE_ACCURACY}% accuracy on 9 people it never saw (UCI HAR). "
              "Real-world results vary with phone placement and sensor calibration.")


# ---------------------------------------------------------------------------
# Activities: one registry for the phone model (UPPER_CASE keys) and the camera model
# (lower-case keys). Colours are CSS classes, never hex.
# ---------------------------------------------------------------------------

@dataclass(frozen=True)
class Act:
    label: str  # "Walking upstairs"
    short: str  # "Upstairs"
    cls: str    # CSS class that sets --act and --on
    icon: str   # key in _ICONS


ACTIVITIES: dict[str, Act] = {
    "WALKING": Act("Walking", "Walking", "act-walking", "walk"),
    "WALKING_UPSTAIRS": Act("Walking upstairs", "Upstairs", "act-upstairs", "up"),
    "WALKING_DOWNSTAIRS": Act("Walking downstairs", "Downstairs", "act-downstairs", "down"),
    "SITTING": Act("Sitting", "Sitting", "act-sitting", "sit"),
    "STANDING": Act("Standing", "Standing", "act-standing", "stand"),
    "LAYING": Act("Lying", "Lying", "act-lying", "lie"),
    "STAIRS": Act("Stairs", "Stairs", "act-upstairs", "up"),  # camera model saved before notebook 09
}
_ALIASES = {"UPSTAIRS": "WALKING_UPSTAIRS", "DOWNSTAIRS": "WALKING_DOWNSTAIRS", "LYING": "LAYING"}
UNKNOWN = Act("Unknown", "Unknown", "act-none", "dot")


def act(key: object) -> Act:
    """Style for 'WALKING', 'walking', 'Walking upstairs' or 'upstairs'. Unknown keys get a neutral style."""
    k = str(key).upper().replace(" ", "_")
    return ACTIVITIES.get(_ALIASES.get(k, k), UNKNOWN)


# Pictograms on a 24-unit grid; stroke, caps and colour come from the .ico CSS class.
_ICONS = {
    "stand": '<circle cx="12" cy="4.5" r="2.1"/><path d="M12 7.6V15M8 12.4l4-3.2 4 3.2M12 15l-2.9 6.2M12 15l2.9 6.2"/>',
    "walk": '<circle cx="13.2" cy="4.5" r="2.1"/><path d="M12.6 7.6l-.9 7.2M12.2 9.6l-3.6 2.8M12.4 9.6l3.8 1.8M11.7 14.8l-3.4 6.4M11.7 14.8l3.9 3 .9 3.4"/>',
    "up": '<path d="M3 20.5h4.5V16H12v-4.5h4.5V7H21M4.5 10.5l5-5M9.5 5.5H6.3M9.5 5.5v3.2"/>',
    "down": '<path d="M3 7h4.5v4.5H12V16h4.5v4.5H21M14.5 4.5l5 5M19.5 9.5h-3.2M19.5 9.5V6.3"/>',
    "sit": '<circle cx="8.5" cy="4.6" r="2.1"/><path d="M8.5 7.7V15M8.5 15h7.2M15.7 15v6M8.5 10.2l4.4 2.6"/>',
    "lie": '<circle cx="4.9" cy="14.6" r="2.1"/><path d="M8 14.6h12.3l1.7 2.4M3 20h18"/>',
    "dot": '<circle cx="12" cy="12" r="3"/>',
    "info": '<circle cx="12" cy="12" r="9"/><path d="M12 11v5M12 8h.01"/>',
}


def icon(key: str, cls: str = "ico") -> str:
    return (f'<svg class="{cls}" viewBox="0 0 24 24" aria-hidden="true" focusable="false">'
            f'{_ICONS.get(key, _ICONS["dot"])}</svg>')


# ---------------------------------------------------------------------------
# Small pure helpers
# ---------------------------------------------------------------------------

def mmss(sec: float) -> str:
    """Seconds as m:ss ('2:04'). One format everywhere."""
    s = max(0, int(round(sec)))
    return f"{s // 60}:{s % 60:02d}"


def listing(words: Sequence[str]) -> str:
    """['a', 'b', 'c'] -> 'a, b and c'."""
    words = list(words)
    return words[0] if len(words) ==

## Tests

### `tests/test_validation.py` · Shravya

In [21]:
show_file("tests/test_validation.py")

`tests/test_validation.py` · 44 lines

import os, sys
import numpy as np, pandas as pd
sys.path.insert(0, os.path.join(os.path.dirname(__file__), "..", "src"))
from validation import validate_csv

CFG = {"sampling_rate_hz": 50, "window_samples": 128}


def make(n=500, scale=1.0, hz=50):
    t = np.arange(n) / hz
    rng = np.random.default_rng(0)
    d = {"timestamp": t}
    for c in ["acc_x", "acc_y", "acc_z"]:
        d[c] = rng.normal(0, 0.05, n) + (1.0 if c == "acc_x" else 0.0)
    for c in ["gyro_x", "gyro_y", "gyro_z"]:
        d[c] = rng.normal(0, 0.1, n)
    df = pd.DataFrame(d)
    df[["acc_x", "acc_y", "acc_z"]] *= scale
    return df


def test_validation():
    r = validate_csv(make(), CFG)
    assert r["ok"] and abs(r["est_hz"] - 50) < 1, r

    bad = make(scale=9.80665)                       # m/s^2 not converted
    bad["timestamp"] = np.where(bad.index >= 250, bad["timestamp"] + 2.0, bad["timestamp"])  # 2 s gap
    r = validate_csv(bad, CFG)
    text = " ".join(r["issues"])
    assert not r["ok"] and "m/s^2" in text and "long gap" in text, r

    short = make()
    short["timestamp"] += (short.index >= 250) * 0.3  # one 0.3 s gap: warning, still ok
    r = validate_csv(short, CFG)
    assert r["ok"] and r["warnings"], r

    assert not validate_csv(make().drop(columns="gyro_z"), CFG)["ok"]
    assert not validate_csv(make(n=100), CFG)["ok"]  # 2 s < 2.56 s


if __name__ == "__main__":
    test_validation()
    print("validation checks passed")

### `tests/test_load_own.py` · Nihal

In [22]:
show_file("tests/test_load_own.py")

`tests/test_load_own.py` · 37 lines

import os, sys, tempfile
import numpy as np, pandas as pd
sys.path.insert(0, os.path.join(os.path.dirname(__file__), "..", "src"))
from data_loader import load_own

CFG = {"sampling_rate_hz": 50, "window_samples": 128, "overlap": 0.5,
       "noise_cutoff_hz": 20, "gravity_cutoff_hz": 0.3}


def rec(seconds=10, hz=100):
    t = np.arange(0, seconds, 1 / hz)
    return pd.DataFrame({"timestamp": t, "acc_x": 1 + 0.1 * np.sin(2 * np.pi * 2 * t), "acc_y": 0.0,
                         "acc_z": 0.0, "gyro_x": 0.0, "gyro_y": 0.0, "gyro_z": 0.0})


def test_load_own():
    with tempfile.TemporaryDirectory() as d:
        rec().to_csv(f"{d}/p1_walking_01.csv", index=False)
        rec().to_csv(f"{d}/p4_lying_01.csv", index=False)
        out = load_own(d, CFG)
        n = len(out["y"]) // 2
        assert out["X"].shape[1:] == (128, 9)
        assert set(out["y"]) == {0, 5} and set(out["subject"]) == {"p1", "p4"}
        assert (out["y"][out["subject"] == "p4"] == 5).all() and n > 0

        rec().to_csv(f"{d}/p2_running_01.csv", index=False)   # unknown activity -> refuse
        try:
            load_own(d, CFG)
            raise AssertionError("bad file name accepted")
        except ValueError as e:
            assert "p2_running_01.csv" in str(e)


if __name__ == "__main__":
    test_load_own()
    print("load_own checks passed")

### `tests/test_smooth.py` · Yashaswini

In [23]:
show_file("tests/test_smooth.py")

`tests/test_smooth.py` · 13 lines

import os, sys
import pandas as pd
sys.path.insert(0, os.path.join(os.path.dirname(__file__), "..", "src"))
from inference import smooth

mk = lambda a: pd.DataFrame({"start_time": range(len(a)), "activity": a, "confidence": 1.0})
src = mk(list("WWSWWW"))
assert list(smooth(src, 5)["activity"]) == list("WWWWWW")      # flicker removed
assert list(src["activity"]) == list("WWSWWW")                  # input untouched
assert list(smooth(mk(list("AABB")), 3)["activity"]) == list("AABB")  # real change kept
assert smooth(mk(list("AB")), 1)["activity"].tolist() == ["A", "B"]  # k=1 is identity
print("ok")

### `tests/test_live.py` · Yashaswini

In [24]:
show_file("tests/test_live.py")

`tests/test_live.py` · 55 lines

import os, sys
from urllib.parse import parse_qsl
import numpy as np
sys.path.insert(0, os.path.join(os.path.dirname(__file__), "..", "src"))
from live import PhyphoxSource


class FakePhone(PhyphoxSource):
    """Answers /get like phyphox: values after the first reference-time above the threshold."""
    def __init__(self):
        super().__init__("192.168.0.42")
        t = np.arange(0, 2, 0.01)
        self.data = {"acc_time": t, "accX": 0 * t, "accY": np.full_like(t, 9.80665), "accZ": 0 * t,
                     "gyr_time": t + 0.005, "gyrX": 0 * t, "gyrY": t, "gyrZ": 0 * t}
        self.now = 1.0  # phone has measured this many seconds so far
        self.commands = []

    def _call(self, path):
        if path.startswith("/control?"):
            self.commands.append(path[13:])
            return {"result": True}
        assert path.startswith("/get?")
        assert "|" not in path, "a real phyphox server returns 400 for an unencoded |"
        out = {}
        for name, v in parse_qsl(path[5:], keep_blank_values=True):
            if not v:  # bare ?acc_time: phyphox returns only the newest value
                out[name] = {"buffer": self.data[name][self.data[name] <= self.now][-1:].tolist()}
                continue
            thr, _, ref = v.partition("|")
            tref = self.data[ref or name]
            keep = (tref > float(thr)) & (tref <= self.now)
            out[name] = {"buffer": self.data[name][keep].tolist()}
        return {"buffer": out}


def test_polls_incrementally_and_converts_units():
    p = FakePhone()
    assert p.base == "http://192.168.0.42:8080"
    a = p.read()
    assert len(a) == 101 and np.allclose(a["acc_x"], 1.0)                 # m/s² -> g
    p.now = 2.0
    b = p.read()
    assert b["timestamp"].iloc[0] > a["timestamp"].iloc[-1]              # no repeats
    assert np.allclose(b["gyro_x"], b["timestamp"] - 0.005, atol=1e-6)   # gyro interpolated onto acc clock


def test_start_skips_old_samples_without_clear():
    p = FakePhone()             # phone already holds 1 s of old data
    p.start()
    assert p.commands == ["start"], "cmd=clear shut down a real phone's web server"
    assert len(p.read()) == 0   # nothing new yet
    p.now = 1.5
    a = p.read()
    assert len(a) == 50 and a["timestamp"].iloc[0] > 1.0

### `tests/test_video_pose.py` · Yyug

In [25]:
show_file("tests/test_video_pose.py")

`tests/test_video_pose.py` · 121 lines

import os, sys, tempfile
import numpy as np
sys.path.insert(0, os.path.join(os.path.dirname(__file__), "..", "src"))
from video_pose import normalize_pose, to_windows, load_video

CFG = {"video": {"fps": 25, "window_frames": 50, "overlap": 0.5, "max_gap": 5,
                  "classes": ["walking", "standing", "stairs"]}}


def pose(n=120, seed=0):
    """Random skeleton, shoulders 2 above hips (torso = 2 after shift), scaled/offset per frame."""
    kp = np.random.default_rng(seed).normal(size=(n, 33, 4)).astype(np.float32)
    kp[:, 23, :2] = [0.4, 0.6]; kp[:, 24, :2] = [0.6, 0.6]
    kp[:, 11, :2] = [0.4, 0.2]; kp[:, 12, :2] = [0.6, 0.2]   # torso length 0.4
    return kp


def test_normalize_centres_and_scales():
    out = normalize_pose(pose())
    assert out.shape == (120, 33, 2)
    assert np.allclose(out[:, [23, 24]].mean(1), 0, atol=1e-6)
    assert np.allclose(np.linalg.norm(out[:, [11, 12]].mean(1), axis=1), 1, atol=1e-6)


def test_gaps():
    kp = pose()
    kp[10:13] = np.nan      # short gap
    kp[40:50] = np.nan      # long gap
    out = normalize_pose(kp, max_gap=5)
    assert not np.isnan(out[10:13]).any()
    assert np.isnan(out[40:50]).all() and not np.isnan(out[:10]).any() and not np.isnan(out[50:]).any()


def test_windows():
    kp = np.zeros((120, 33, 2))
    w = to_windows(kp, 50, 0.5)
    assert w.shape == (3, 50, 66)       # starts 0, 25, 50 (70 would overflow)
    kp[60] = np.nan                     # hits windows starting at 25 and 50
    assert to_windows(kp, 50, 0.5).shape == (1, 50, 66)


def test_load_video():
    with tempfile.TemporaryDirectory() as d:
        np.savez(f"{d}/p1_walking_01.npz", kp=pose(), fps=25)
        np.savez(f"{d}/p4_stairs_02.npz", kp=pose(seed=1), fps=25)
        out = load_video(d, CFG)
        assert out["X"].shape[1:] == (50, 66) and len(out["X"]) == len(out["y"]) == len(out["subject"])
        assert (out["y"][out["subject"] == "p1"] == 0).all() and (out["y"][out["subject"] == "p4"] == 2).all()
        np.savez(f"{d}/p2_running_01.npz", kp=pose(), fps=25)
        try:
            load_video(d, CFG)
            raise AssertionError("bad file name accepted")
        except ValueError as e:
            assert "p2_running_01.npz" in str(e)


def test_pose_features():
    from video_pose import pose_features, FEATURE_NAMES
    t = np.arange(50) / 25
    still = normalize_pose(np.repeat(pose()[:1], 50, axis=0))[..., :].reshape(50, 66)
    walk = normalize_pose(np.repeat(pose()[:1], 50, axis=0)).copy()
    walk[:, 27, 1] += 0.3 * np.sin(2 * np.pi * 1.0 * t)          # left ankle swings at 1 Hz
    walk[:, 28, 1] -= 0.3 * np.sin(2 * np.pi * 1.0 * t)
    a, b = pose_features(still), pose_features(walk.reshape(50, 66))
    assert a.shape == b.shape == (len(FEATURE_NAMES),) and np.isfinite(b).all()
    i = FEATURE_NAMES.index
    assert b[i("spd_ankle")] > a[i("spd_ankle")] + 0.5
    assert abs(b[i("ankle_dy_peak_hz")] - 1.0) < 0.2 and b[i("ankle_dy_peak_frac")] > 0.5


def test_predict_live():
    from video_pose import predict_live

    class Echo:  # fake model: probability of class 0 = mean of the first feature, kept in 0-1
        def predict_proba(self, X):
            p = 1 / (1 + np.exp(-X[:, :1]))
            return np.hstack([p, 1 - p])

    bundle = {"model": Echo(), "classes": ["walking", "standing"]}
    cfg = {"video": dict(CFG["video"])}
    kp = pose(90)
    t30 = np.arange(90) / 30 + np.random.default_rng(1).uniform(0, 0.005, 90)  # jittery 30 fps webcam
    assert predict_live(t30[:40], kp[:40], bundle, cfg) is None      # 1.3 s: shorter than the 2 s window
    act, conf = predict_live(t30, kp, bundle, cfg)
    assert act in ("walking", "standing") and 0.5 <= conf <= 1
    kp[-40:] = np.nan                                                  # person left the frame
    assert predict_live(t30, kp, bundle, cfg) is None


def test_stairs_become_up_or_down():
    from video_pose import _classify

    class Fixed:  # fake model: always the given probabilities
    

### `tests/test_ui_helpers.py` · Yashaswini

In [26]:
show_file("tests/test_ui_helpers.py")

`tests/test_ui_helpers.py` · 31 lines

"""UI helper checks: every activity key a model can emit has a style, and the summary line is honest."""
import pathlib
import sys

ROOT = pathlib.Path(__file__).resolve().parents[1]
sys.path.insert(0, str(ROOT / "app"))
sys.path.insert(0, str(ROOT / "src"))

from components.ui import act, coalesce, mmss, summarize  # noqa: E402
from inference import ACTS  # noqa: E402

CAMERA_KEYS = ["walking", "standing", "upstairs", "downstairs", "stairs"]


def test_every_model_activity_has_a_style():
    for key in list(ACTS) + CAMERA_KEYS:
        assert act(key).cls != "act-none", key


def test_summary_only_says_mostly_when_it_dominates():
    assert summarize({"WALKING": 7, "SITTING": 3})[0] == "Mostly walking"
    mixed = {"WALKING": 21, "SITTING": 21, "STANDING": 21, "LAYING": 19,
             "WALKING_UPSTAIRS": 17, "WALKING_DOWNSTAIRS": 19}
    assert summarize(mixed)[0] == "A mix of six activities"
    assert summarize({"STANDING": 5})[0] == "All standing"


def test_time_format_and_runs():
    assert mmss(124) == "2:04"
    assert coalesce(["A", "A", "B", "A"]) == [("A", 2), ("B", 1), ("A", 1)]